# 17. XGB 시드 평균 + LR 섞기 (H17)

## 배경
- 14: XGB 0.7 + LR 0.3 섞기 → **LB 0.3247 (현재 최고)**, OOF 0.4231
- XGB는 `colsample_bytree=0.3`이라 나무마다 피처의 30%만 무작위로 봄 → `random_state`(시드)만 바꿔도 예측이 달라짐
- 섞기 개선 폭(+0.006)이 XGB std(0.006)와 비슷 → 시드 하나의 운에 따라 결과가 흔들릴 수 있음

## 가설
- H17: XGB를 시드 5개로 학습해 확률을 평균한 뒤 LR과 섞으면, 시드 하나의 운에 좌우되지 않고(시드 운 제거) 점수가 같거나 높아진다
- H0: 시드 평균은 시드 하나와 차이가 없다

## 검증 방법
- 데이터·피처·설정은 14와 동일: `load_data()` (정규화), XGB = any + stats (10 설정), LR = any (C=0.1)
- **fold 분할은 팀 공통(seed 42) 그대로**, XGB의 `random_state`만 5개(42, 0, 1, 2, 3)로 바꿈
- 섞는 비율은 14에서 고른 0.7 / 0.3 고정 (다시 고르지 않음 → OOF 과적합 방지)
- 비교: 시드별 섞기 5개 vs 시드 5개 평균 섞기 (fold 평균 CV, OOF, fold별 짝 비교)
- 판단: 시드 평균이 시드별 섞기의 평균 이상이고 fold 5개 중 4개 이상에서 시드 42보다 높으면 제출 후보

#### 준비 (14와 같은 데이터·피처·모델 설정)

In [3]:
import sys; sys.path.append('..')
import time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, run_cv, get_folds, macro_f1, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
X_any,  X_any_test  = F_tr['any'].astype('float32'), F_te['any'].astype('float32')
X_tree, X_tree_test = pd.concat([F_tr['any'], F_tr['stats']], axis=1), pd.concat([F_te['any'], F_te['stats']], axis=1)
folds = get_folds(y)   # 팀 공통 분할 (seed 42) → 모든 시드가 같은 fold

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

def make_xgb(seed):   # 10 설정, random_state만 바꿈
    return lambda: BalancedXGB(n_estimators=200, learning_rate=0.1, max_depth=3, colsample_bytree=0.3,
                               tree_method='hist', eval_metric='mlogloss', random_state=seed, n_jobs=-1)

def fold_scores(P):
    return [macro_f1(y[va], P[va].argmax(1)) for _, va in folds]

#### 실행 환경 확인 (리뷰 요청)
- `load_data()`가 정규화 버전인지(`dedupe` 옵션 유무), 라이브러리 버전, CPU 종류를 기록
- XGB CV가 실행 환경마다 조금씩 달라서 비교할 때 참고


In [8]:
import inspect, platform, sklearn
print('정규화 버전 load_data (dedupe 옵션):', 'dedupe' in str(inspect.signature(load_data)))
print('xgboost', xgb.__version__, '| scikit-learn', sklearn.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
print('플랫폼:', platform.platform(), '|', platform.machine())


정규화 버전 load_data (dedupe 옵션): True
xgboost 3.4.1 | scikit-learn 1.9.1 | numpy 2.5.4 | pandas 3.1.1
플랫폼: macOS-26.5.2-arm64-arm-64bit | arm64


#### 모델별 OOF·test 확률 계산
- LR은 무작위성이 없어 1번만 학습
- XGB는 시드 5개 각각 학습 (시드 1개당 약 3분)

In [4]:
r = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
           X_any, y, X_any_test, folds=folds, verbose=False)
LR_oof, LR_test = r['oof_proba'], r['test_proba']
print(f"LR: CV {r['cv_mean']:.4f} ± {r['cv_std']:.4f}")

SEEDS = [42, 0, 1, 2, 3]
XGB_oof, XGB_test = {}, {}
for s in SEEDS:
    t = time.time()
    r = run_cv(make_xgb(s), X_tree, y, X_tree_test, folds=folds, verbose=False)
    XGB_oof[s], XGB_test[s] = r['oof_proba'], r['test_proba']
    print(f"XGB seed {s}: CV {r['cv_mean']:.4f} ± {r['cv_std']:.4f} ({time.time() - t:.0f}s)")

LR: CV 0.3818 ± 0.0065
XGB seed 42: CV 0.4125 ± 0.0109 (168s)
XGB seed 0: CV 0.4126 ± 0.0089 (170s)
XGB seed 1: CV 0.4107 ± 0.0074 (165s)
XGB seed 2: CV 0.4145 ± 0.0056 (166s)
XGB seed 3: CV 0.4133 ± 0.0063 (167s)


#### 시드별 섞기 vs 시드 평균 섞기
- 섞기 = XGB 0.7 + LR 0.3 (14와 같은 비율)
- 시드 평균 = XGB 시드 5개의 확률을 먼저 평균 → 그다음 LR과 섞기

In [5]:
W = 0.7
rows = []
for s in SEEDS:
    fs = fold_scores(W * XGB_oof[s] + (1 - W) * LR_oof)
    rows.append(dict(name=f'시드 {s}', cv=np.mean(fs), std=np.std(fs),
                     oof=macro_f1(y, (W * XGB_oof[s] + (1 - W) * LR_oof).argmax(1))))
XGB_avg_oof = np.mean([XGB_oof[s] for s in SEEDS], axis=0)
XGB_avg_test = np.mean([XGB_test[s] for s in SEEDS], axis=0)
P_avg_oof = W * XGB_avg_oof + (1 - W) * LR_oof
fs_avg = fold_scores(P_avg_oof)
rows.append(dict(name='시드 5개 평균', cv=np.mean(fs_avg), std=np.std(fs_avg), oof=macro_f1(y, P_avg_oof.argmax(1))))
res17 = pd.DataFrame(rows).round(4)
single = res17.iloc[:-1]
print(f"시드별 섞기: 평균 {single['cv'].mean():.4f}, 범위 {single['cv'].min():.4f} ~ {single['cv'].max():.4f}")
res17

시드별 섞기: 평균 0.4210, 범위 0.4187 ~ 0.4237


,name,cv,std,oof
0,시드 42,0.4187,0.0065,0.4190
1,시드 0,0.4197,0.0066,0.4203
2,시드 1,0.4199,0.0092,0.4202
3,시드 2,0.4230,0.0053,0.4236
4,시드 3,0.4237,0.0048,0.4242
5,시드 5개 평균,0.4220,0.0079,0.4223


#### fold별 짝 비교 + test 예측 변화
- 같은 fold라 fold마다 직접 비교 가능
- test 예측이 시드에 따라 얼마나 바뀌는지 (시드 운의 크기), 시드 평균이 시드 42 대비 얼마나 바뀌는지

In [6]:
fs42 = fold_scores(W * XGB_oof[42] + (1 - W) * LR_oof)
print('시드 평균 > 시드 42 fold 수:', sum(a > b for a, b in zip(fs_avg, fs42)), '/ 5')

pred = {s: (W * XGB_test[s] + (1 - W) * LR_test).argmax(1) for s in SEEDS}
pred_avg = (W * XGB_avg_test + (1 - W) * LR_test).argmax(1)
pair = [np.mean(pred[a] != pred[b]) for i, a in enumerate(SEEDS) for b in SEEDS[i + 1:]]
print(f'시드끼리 test 예측 불일치율 (평균): {np.mean(pair):.1%}')
print(f'시드 평균 vs 시드 42: {np.mean(pred_avg != pred[42]):.1%} ({int(np.sum(pred_avg != pred[42]))}명)')

pd.DataFrame({'fold': range(5), '시드 42 섞기': np.round(fs42, 4), '시드 평균 섞기': np.round(fs_avg, 4)})

시드 평균 > 시드 42 fold 수: 4 / 5
시드끼리 test 예측 불일치율 (평균): 6.9%
시드 평균 vs 시드 42: 4.0% (103명)


,fold,시드 42 섞기,시드 평균 섞기
0,0,0.4113,0.4176
1,1,0.4198,0.4234
2,2,0.4267,0.4291
3,3,0.4109,0.4091
4,4,0.4246,0.4305


#### test 예측 분포 (관찰용)
- 시드 42 섞기와 시드 평균 섞기의 아형별 예측 비율 상위 6개 (test 라벨은 쓰지 않음, 판단 기준 아님)


In [9]:
dist = pd.DataFrame({'시드 42 섞기': pd.Series(le.classes_[pred[42]]).value_counts(normalize=True),
                     '시드 평균 섞기': pd.Series(le.classes_[pred_avg]).value_counts(normalize=True)}).fillna(0)
(dist.sort_values('시드 평균 섞기', ascending=False).head(6) * 100).round(1).astype(str) + '%'


,시드 42 섞기,시드 평균 섞기
COAD,16.3%,17.1%
STES,11.7%,11.3%
OV,5.3%,5.3%
TGCT,4.8%,4.9%
BRCA,5.1%,4.9%
KIPAN,4.1%,4.1%


#### 제출 파일 생성
- 비율은 14와 같은 0.7 / 0.3 (다시 고르지 않음)

In [7]:
P_avg_test = W * XGB_avg_test + (1 - W) * LR_test
save_submission(submission, P_avg_test, le, name='blend_XGB5seed0.7_LR0.3', cv_score=macro_f1(y, P_avg_oof.argmax(1)))

저장 완료: /Users/mac/web/dacon-cancer-subclass/submissions/1111_blend_XGB5seed0.7_LR0.3_0.422.csv


PosixPath('/Users/mac/web/dacon-cancer-subclass/submissions/1111_blend_XGB5seed0.7_LR0.3_0.422.csv')

## 결과

| 섞기 (XGB 0.7 + LR 0.3) | CV (± std) | OOF |
| :--- | :-: | :-: |
| 시드 42 | 0.4187 ± 0.0065 | 0.4190 |
| 시드 0 | 0.4197 ± 0.0066 | 0.4203 |
| 시드 1 | 0.4199 ± 0.0092 | 0.4202 |
| 시드 2 | 0.4230 ± 0.0053 | 0.4236 |
| 시드 3 | 0.4237 ± 0.0048 | 0.4242 |
| **시드 5개 평균** | **0.4220 ± 0.0079** | **0.4223** |

- **시드만 바꿔도 섞기 CV가 0.4187 ~ 0.4237로 0.005 흔들림** → 14의 섞기 개선 폭(+0.006)과 비슷한 크기
- 시드 평균(0.4220)은 시드별 평균(0.4210)과 비슷하거나 약간 높음, 시드 42보다 fold 5개 중 4개에서 높음
- test 예측: 시드끼리 평균 6.9%가 다르게 예측 → 시드 평균은 시드 42 대비 4.0%(103명) 변경
- 예측 분포는 거의 같음 (COAD 16.3% → 17.1%, STES 11.7% → 11.3%, "test 예측 분포" 셀)
- **실행 환경 차이 (리뷰 요청 확인)**: 이 환경의 XGB 시드 42 CV 0.4125 vs 팀 기록 0.4161 (14, Windows)
  - `load_data()`는 정규화 버전 (dedupe 옵션 있음, 로컬 `src/`가 최신 main과 동일)
  - 라이브러리 버전도 requirements.txt와 같음 (xgboost 3.4.1, scikit-learn 1.9.1, numpy 2.5.4, pandas 3.1.1)
  - 같은 코드로 정규화 전 데이터(`normalize=False`)를 돌리면 이 환경은 0.4110 ± 0.0102 vs 팀 10번 0.4113 ± 0.0024 → 평균은 비슷하지만 **fold별 점수가 다름** → 정규화와 무관하게 실행 환경 자체에서 차이가 남
  - 원인은 CPU 종류 차이로 추정 (이 환경 Apple M1 arm64 / 팀 Windows x86): XGBoost `hist`의 실수 합산 순서가 달라 분기가 조금씩 바뀜. 같은 환경에서는 여러 번 돌려도 결과가 완전히 같음
  - 크기: 환경 차이(0.0036) ≈ 시드 차이(0.4107 ~ 0.4145) → 환경 차이도 "시드 운"과 같은 종류의 흔들림
  - 같은 노트북 안의 비교(시드 42 vs 시드 평균)는 같은 환경이라 공정함

## 결론
- **H17 채택 (시드 운 제거 기준)**: 시드 평균은 점수를 크게 올리기보다 **시드 하나의 운에 좌우되지 않게** 하는 효과 (시드별 0.4187 ~ 0.4237 → 평균 0.4220)
  - fold 간 std는 오히려 시드 42보다 큼(0.0079 vs 0.0065) → "fold마다 고르게 맞힌다"는 의미의 안정성은 아님
- 개선 폭이 std 이내라 CV로는 "같은 점수", 최종 판단은 LB
- LB: 미제출. 11/11 마지막 제출 여부는 회의에서 결정
  - 제출한다면 비교 대상은 14 XGB 0.7 + LR 0.3 (LB 0.3247)
  - ⚠️ 이 제출 파일은 이 환경(M1)에서 만들어져 0.3247(Windows)과 환경이 다름 → LB 차이를 시드 평균 효과로만 해석할 수 없음. 시드 평균 효과를 깔끔하게 보려면 0.3247과 같은 환경에서 이 노트북을 다시 실행해 제출

## 다음
- 최종 제출 조합(14·16 후보)에도 같은 방식으로 XGB 시드 평균 적용 가능 → 재현 스크립트에 시드 평균 옵션으로 포함